In [2]:
import sys; sys.path.append("..")
import pandas as pd
from collections import Counter
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from src.data import load_data
from src.config import RANDOM_STATE
from src.experiment_log import ExperimentLogger

X_train, _, y_train, _, labels = load_data()
lg = ExperimentLogger("thai")

def under_cap(y, cap=5000):          # giam lop lon xuong toi da 5000 mau
    return {k: cap for k, v in Counter(y).items() if v > cap}

def smote_to(y, target=2000):        # nang lop hiem len 2000 mau
    return {k: target for k, v in Counter(y).items() if v < target}

def rf_factory():
    return RandomForestClassifier(n_estimators=100, max_depth=10, max_features="log2",
                                  n_jobs=-1, random_state=RANDOM_STATE)

def hgb_factory(**kw):
    return HistGradientBoostingClassifier(early_stopping=False,
                                          random_state=RANDOM_STATE, **kw)   # thay bang tham so hgb_tuned neu co

def make_variants(factory):
    us = lambda: RandomUnderSampler(sampling_strategy=under_cap, random_state=RANDOM_STATE)
    sm = lambda: SMOTE(sampling_strategy=smote_to, k_neighbors=3, random_state=RANDOM_STATE)
    return {
        "N2_under":       ImbPipeline([("us", us()), ("clf", factory())]),
        "N3_smote":       ImbPipeline([("sc", StandardScaler()), ("sm", sm()), ("clf", factory())]),
        "N4_under_smote": ImbPipeline([("us", us()), ("sc", StandardScaler()), ("sm", sm()), ("clf", factory())]),
    }

# N0 va N1 cho RF: lay tu log cua B (rf_d10_cw_None, rf_d10_cw_balanced), khong chay lai
# N0 cho HGB: hgb_tuned (da co tu Buoc 4); N1 cho HGB:
lg.run(hgb_factory(class_weight="balanced"), X_train, y_train, "hgb_N1_balanced",
       strategy="class_weight=balanced", notes="N1 tren HGB")

for base, factory in [("rf", rf_factory), ("hgb", hgb_factory)]:
    for vname, pipe in make_variants(factory).items():
        lg.run(pipe, X_train, y_train, f"{base}_{vname}",
               strategy=f"{vname} (cap=5000 / SMOTE target=2000, k=3)",
               notes="thi nghiem mat can bang")